# Endpoints da API do Portal da Transparência MG

Uma célula por endpoint de [api_consultas_portal_transparencia.yaml](../docs/api_consultas_portal_transparencia.yaml),
com a resposta real da API de homologação. Usa a credencial do `.env` através do
[client.py](../mcp_portal_transparencia/client.py) (token OAuth2 renovado automaticamente).

Como ler as saídas:
- a primeira linha mostra método, path, **status HTTP**, tempo e tamanho;
- listas longas aparecem cortadas nos 3 primeiros itens, com `… (+N itens)` indicando quantos faltam;
- cada função `chamar(...)` devolve o JSON completo, para encadear (ex.: o `id_empenho` de uma célula vai para a próxima).

Referência dos comportamentos: [ENDPOINTS.md](../docs/ENDPOINTS.md).

In [ ]:
import csv, io, json, sys, time
from pathlib import Path

# acha a raiz do projeto (pasta que contem mcp_portal_transparencia/) a partir de onde o notebook roda
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mcp_portal_transparencia").is_dir())
sys.path.insert(0, str(RAIZ / "mcp_portal_transparencia"))
import client

ANO = 2025
P = {"ano": ANO, "periodo_inicio": "0101", "periodo_fim": "3112"}   # despesas e diárias: DDMM
PAUSA = 0.5  # segundos entre chamadas, para não sobrecarregar o gateway


def resumir(obj, max_itens=3):
    """Corta listas longas para a saída caber na tela."""
    if isinstance(obj, dict):
        return {k: resumir(v, max_itens) for k, v in obj.items()}
    if isinstance(obj, list):
        cortada = [resumir(v, max_itens) for v in obj[:max_itens]]
        return cortada + [f"… (+{len(obj) - max_itens} itens)"] if len(obj) > max_itens else cortada
    return obj


def chamar(metodo, path, params=None, corpo=None, max_itens=3, max_linhas=60):
    """Faz a requisição, imprime um resumo e devolve o JSON (ou os bytes, se não for JSON)."""
    time.sleep(PAUSA)
    inicio = time.time()
    try:
        r = client._http.request(metodo, path, params=params, json=corpo, timeout=120,
                                 headers={"Authorization": f"Bearer {client._obter_token()}"})
    except Exception as e:
        print(f"{metodo} {path} -> ERRO de conexão: {e}")
        return None
    tipo = r.headers.get("content-type", "")
    print(f"{metodo} {path} -> HTTP {r.status_code}  ({time.time() - inicio:.1f} s, {len(r.content):,} bytes, {tipo})")
    if params:
        print("params:", params)
    if corpo:
        print("corpo: ", json.dumps(corpo, ensure_ascii=False))
    print()
    if "json" in tipo:
        dados = r.json()
        texto = json.dumps(resumir(dados, max_itens), ensure_ascii=False, indent=2).splitlines()
        print("\n".join(texto[:max_linhas]) + (f"\n… ({len(texto) - max_linhas} linhas omitidas)" if len(texto) > max_linhas else ""))
        return dados
    if "csv" in tipo:
        linhas = r.content.decode("utf-8-sig", "replace").splitlines()
        print(f"CSV com {len(linhas)} linhas. Cabeçalho e primeiras linhas:")
        print("\n".join(l[:300] for l in linhas[:3]))
    elif r.content[:2] == b"PK":
        print("Arquivo binário (.xlsx/.zip); conteúdo não exibido.")
    else:
        print(r.text[:1000])
    return r.content


def achar(obj, *chaves):
    """Primeiro valor não vazio de uma das chaves, procurando em qualquer profundidade."""
    if isinstance(obj, dict):
        for k in chaves:
            if obj.get(k) not in (None, ""):
                return obj[k]
        obj = list(obj.values())
    if isinstance(obj, list):
        for v in obj:
            achado = achar(v, *chaves)
            if achado is not None:
                return achado
    return None


print("Base:", client.BASE_URL)

Base: https://homologacao.apigw.mgapi.prodemge.gov.br/portaltransparencia/1.0.0


In [ ]:
# Ids de 2025 usados nas consultas (os ids mudam a cada ano; os códigos não)
ID_AGE = 12135          # 1081 - ADVOCACIA GERAL DO ESTADO
ID_SEE = 12158          # 1261 - SECRETARIA DE ESTADO DE EDUCACAO
ID_FES = 12108          # 4291 - FUNDO ESTADUAL DE SAUDE
ID_ACAO_PEAE = 75488    # ação 2101 - Programa Estadual de Alimentação Escolar
ID_ELEMENTO_MAT_CONSUMO = 22  # elemento 30 - MATERIAL DE CONSUMO
ID_FUNCAO_SAUDE = 4711  # função 10 - SAUDE
ID_SUBFUNCAO_ALIMENTACAO = 5143  # subfunção 306 - ALIMENTACAO E NUTRICAO

## 1. Gerais

### `GET /consultas/consultas.json`
Consultas cadastradas no portal, por tipo.

In [ ]:
r = chamar("GET", "/consultas/consultas.json", {"tipo": "despesas", "limit": 5, "page": 1})

GET /consultas/consultas.json -> HTTP 200  (0.5 s, 398 bytes, application/json)
params: {'tipo': 'despesas', 'limit': 5, 'page': 1}

{
  "status": 200,
  "data": {
    "consultas": [
      {
        "id": 26,
        "titulo": "Despesas",
        "tipo_consulta": "despesas",
        "slug_consulta": "despesas",
        "slug_categoria": "despesas-publicas-1",
        "slug_frontend": "consultas/despesas"
      }
    ],
    "totalRegistros": 1
  }
}


### `GET /consultas/data-ultima-carga.json`
Data da última carga de cada módulo. Para `diarias` a API responde 204 (sem conteúdo).

In [ ]:
for tipo in ["despesas", "receitas", "diarias", "remuneracao"]:
    r = chamar("GET", "/consultas/data-ultima-carga.json", {"tipo": tipo})
    print("-" * 80)

GET /consultas/data-ultima-carga.json -> HTTP 200  (0.2 s, 65 bytes, application/json)
params: {'tipo': 'despesas'}

{
  "status": 200,
  "data": [
    "20/10/2026"
  ]
}
--------------------------------------------------------------------------------


GET /consultas/data-ultima-carga.json -> HTTP 200  (0.1 s, 65 bytes, application/json)
params: {'tipo': 'receitas'}

{
  "status": 200,
  "data": [
    "20/10/2026"
  ]
}
--------------------------------------------------------------------------------


GET /consultas/data-ultima-carga.json -> HTTP 204  (0.2 s, 0 bytes, )
params: {'tipo': 'diarias'}


--------------------------------------------------------------------------------


GET /consultas/data-ultima-carga.json -> HTTP 200  (0.2 s, 65 bytes, application/json)
params: {'tipo': 'remuneracao'}

{
  "status": 200,
  "data": [
    "21/08/2026"
  ]
}
--------------------------------------------------------------------------------


## 2. Despesas: consultas de valores

Período em **DDMM** (`0101`–`3112` = ano inteiro). Os `total_*` valem para o conjunto filtrado inteiro, não só para a página.

### `GET /consultas/despesas/lista-periodo.json`
Anos disponíveis.

In [ ]:
r = chamar("GET", "/consultas/despesas/lista-periodo.json")

GET /consultas/despesas/lista-periodo.json -> HTTP 200  (3.1 s, 169 bytes, application/json)

{
  "status": 200,
  "data": [
    [
      2021,
      2022,
      2023,
      "… (+3 itens)"
    ]
  ]
}


### `GET /consultas/despesas/visao-geral.json`
Despesa por órgão. ⚠️ Ignora `id_funcao`, `id_subfuncao` e `id_elemento_despesa` sem avisar.

In [ ]:
r = chamar("GET", "/consultas/despesas/visao-geral.json", {**P, "limit": 3, "page": 1})

GET /consultas/despesas/visao-geral.json -> HTTP 200  (1.5 s, 3,130 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "total_empenhado": 131563975828.04,
    "total_liquidado": 127112592180.39,
    "total_pago": 125459430296.87,
    "total_liquidado_restos_a_pagar": 3350312793.92,
    "total_pago_restos_a_pagar": 4431039657.47,
    "subtotal_empenhado": 1036465271.46,
    "subtotal_liquidado": 1026201928.52,
    "subtotal_pago": 770411925.15,
    "subtotal_liquidado_restos_a_pagar": 2981676.46,
    "subtotal_pago_restos_a_pagar": 14312328.4,
    "qt_orgaos": 87,
    "orgaos": [
      {
        "id_orgao": 12135,
        "cd_orgao": 1081,
        "sigla": "AGE",
        "nome": "ADVOCACIA GERAL DO ESTADO",
        "empenhado": 1023743951.58,
        "liquidado": 1014541849.87,
        "pago": 759567117.27,
        "liquidado_restos_a_pagar": 2761361.24,
        "pago_restos_a_pagar":

### `POST /consultas/despesas/pesquisa-avancada.json`
Filtros combinados; o único que respeita função e subfunção. `ano` e período **escalares**; ids aceitam lista. Só funciona com `POST` e `.json`.

In [ ]:
r = chamar("POST", "/consultas/despesas/pesquisa-avancada.json",
           corpo={"nivel": 1, "filtros": {**P, "id_funcao": ID_FUNCAO_SAUDE}, "limit": 5, "page": 1})

POST /consultas/despesas/pesquisa-avancada.json -> HTTP 200  (0.7 s, 6,664 bytes, application/json)
corpo:  {"nivel": 1, "filtros": {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "id_funcao": 4711}, "limit": 5, "page": 1}

{
  "status": 200,
  "message": "Os resultados da consulta foram atualizados.",
  "data": {
    "nivel": 1,
    "total": 11,
    "page": 1,
    "limit": 5,
    "total_paginas": 3,
    "order_by": "orgao",
    "order_dir": "DESC",
    "periodo": "01/01/25 – 31/12/25",
    "quantidade_filtros_aplicados": 2,
    "filtros_aplicados": {
      "ano": [
        2025
      ],
      "periodo_inicio": [
        "0101"
      ],
      "periodo_fim": [
        "3112"
      ],
      "id_funcao": [
        4711
      ]
    },
    "colunas_disponiveis": [
      {
        "campo": "periodo",
        "titulo": "Período",
        "padrao": false
      },
      {
        "campo": "orgao",
        "titulo": "Órgão",
        "padrao": true,
        "fixa": true
      },
  

Agrupando por função e subfunção com `colunas` (a API agrupa sempre também por órgão):

In [ ]:
r = chamar("POST", "/consultas/despesas/pesquisa-avancada.json",
           corpo={"nivel": 1, "filtros": {**P, "id_funcao": ID_FUNCAO_SAUDE},
                  "colunas": "funcao,subfuncao,valor_pago", "order_by": "valor_pago", "order_dir": "DESC",
                  "limit": 5, "page": 1})

POST /consultas/despesas/pesquisa-avancada.json -> HTTP 200  (0.8 s, 5,777 bytes, application/json)
corpo:  {"nivel": 1, "filtros": {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "id_funcao": 4711}, "colunas": "funcao,subfuncao,valor_pago", "order_by": "valor_pago", "order_dir": "DESC", "limit": 5, "page": 1}

{
  "status": 200,
  "message": "Os resultados da consulta foram atualizados.",
  "data": {
    "nivel": 1,
    "total": 28,
    "page": 1,
    "limit": 5,
    "total_paginas": 6,
    "order_by": "valor_pago",
    "order_dir": "DESC",
    "periodo": "01/01/25 – 31/12/25",
    "quantidade_filtros_aplicados": 2,
    "filtros_aplicados": {
      "ano": [
        2025
      ],
      "periodo_inicio": [
        "0101"
      ],
      "periodo_fim": [
        "3112"
      ],
      "id_funcao": [
        4711
      ]
    },
    "colunas_disponiveis": [
      {
        "campo": "periodo",
        "titulo": "Período",
        "padrao": false
      },
      {
        "campo"

### `GET /consultas/despesas/programas.json`
Despesa por órgão + programa + ação (uma linha por combinação).

In [ ]:
r = chamar("GET", "/consultas/despesas/programas.json", {**P, "id_acao": ID_ACAO_PEAE, "limit": 3, "page": 1})

GET /consultas/despesas/programas.json -> HTTP 200  (1.1 s, 2,767 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'id_acao': 75488, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "total_empenhado": 519971890.05,
    "total_liquidado": 519835138.05,
    "total_pago": 518660591.29,
    "total_liquidado_restos_a_pagar": 0,
    "total_pago_restos_a_pagar": 0,
    "subtotal_empenhado": 519971890.05,
    "subtotal_liquidado": 519835138.05,
    "subtotal_pago": 518660591.29,
    "subtotal_liquidado_restos_a_pagar": 0,
    "subtotal_pago_restos_a_pagar": 0,
    "qt_programas": 1,
    "programas": [
      {
        "acao": "PROGRAMA ESTADUAL DE ALIMENTACAO ESCOLAR - PEAE",
        "empenhado": 519971890.05,
        "liquidado": 519835138.05,
        "pago": 518660591.29,
        "liquidado_restos_a_pagar": 0,
        "pago_restos_a_pagar": 0
      }
    ],
    "colunas_disponiveis": [
      {
        "campo": "codigo_acao",
        "ti

### `GET /consultas/despesas/itens-despesa.json`
Despesa por item e elemento de despesa.

In [ ]:
r = chamar("GET", "/consultas/despesas/itens-despesa.json",
           {**P, "id_orgao": ID_FES, "id_elemento_despesa": ID_ELEMENTO_MAT_CONSUMO, "limit": 3, "page": 1})

GET /consultas/despesas/itens-despesa.json -> HTTP 200  (1.9 s, 3,916 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'id_orgao': 12108, 'id_elemento_despesa': 22, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "total_empenhado": 905580847.74,
    "total_liquidado": 676777421.54,
    "total_pago": 644278655.85,
    "total_liquidado_restos_a_pagar": 146655252.56,
    "total_pago_restos_a_pagar": 197085157.24,
    "subtotal_empenhado": 1785,
    "subtotal_liquidado": 500,
    "subtotal_pago": 500,
    "subtotal_liquidado_restos_a_pagar": 0,
    "subtotal_pago_restos_a_pagar": 0,
    "qt_itens_despesa": 25,
    "itens_despesa": [
      {
        "id_orgao": 12108,
        "item_despesa": "MATERIAL PARA ESCRITORIO",
        "elemento_despesa": "MATERIAL DE CONSUMO",
        "orgao": "FUNDO ESTADUAL DE SAUDE",
        "empenhado": 370,
        "liquidado": 0,
        "pago": 0,
        "liquidado_restos_a_pagar": 0,
        "pago_r

### `GET /consultas/despesas/favorecidos.json`
Despesa por credor. ⚠️ Sem `colunas` não traz valores; `totais` vêm vazios.

In [ ]:
r = chamar("GET", "/consultas/despesas/favorecidos.json",
           {**P, "id_orgao": ID_SEE, "colunas": "favorecido,cnpj_cpf_favorecido,valor_empenhado,valor_pago",
            "order_by": "valor_pago", "order_dir": "DESC", "limit": 5, "page": 1})

GET /consultas/despesas/favorecidos.json -> HTTP 200  (3.0 s, 4,105 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'id_orgao': 12158, 'colunas': 'favorecido,cnpj_cpf_favorecido,valor_empenhado,valor_pago', 'order_by': 'valor_pago', 'order_dir': 'DESC', 'limit': 5, 'page': 1}

{
  "status": 200,
  "data": {
    "colunas_disponiveis": [
      {
        "campo": "favorecido",
        "titulo": "Favorecido",
        "padrao": true,
        "fixa": true
      },
      {
        "campo": "cnpj_cpf_favorecido",
        "titulo": "CPF/CNPJ Favorecido",
        "padrao": false
      },
      {
        "campo": "codigo_orgao",
        "titulo": "Código do Órgão",
        "padrao": false
      },
      "… (+9 itens)"
    ],
    "colunas_exibidas": [
      "favorecido",
      "cnpj_cpf_favorecido",
      "valor_empenhado",
      "… (+1 itens)"
    ],
    "qt_registros": 21517,
    "registros": [
      {
        "id_orgao": 12158,
        "id_favorec

### `GET /consultas/despesas/empenhos.json`
Lista de empenhos. ⚠️ Exige `id_orgao`, `id_unidade` ou `favorecido`; pode levar ~20 s.

In [ ]:
empenhos = chamar("GET", "/consultas/despesas/empenhos.json",
                  {**P, "id_orgao": ID_AGE, "order_by": "empenhado", "order_dir": "DESC", "limit": 3, "page": 1})
ID_EMPENHO = achar(empenhos, "id_empenho")
print("\nid_empenho usado nas próximas células:", ID_EMPENHO)

GET /consultas/despesas/empenhos.json -> HTTP 200  (16.3 s, 6,703 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'id_orgao': 12135, 'order_by': 'empenhado', 'order_dir': 'DESC', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "total_empenhado": 1023743951.58,
    "total_liquidado": 1014541849.87,
    "total_pago": 759567117.27,
    "total_liquidado_restos_a_pagar": 2761925.39,
    "total_pago_restos_a_pagar": 14037404.22,
    "subtotal_empenhado": 378832431.02,
    "subtotal_liquidado": 378832431.02,
    "subtotal_pago": 369810085.68,
    "subtotal_liquidado_restos_a_pagar": 0,
    "subtotal_pago_restos_a_pagar": 0,
    "qt_empenhos": 116424,
    "empenhos": [
      {
        "id_empenho": 17475715,
        "id_orgao": 12135,
        "id_unidade_executora": 38049,
        "empenho": 90,
        "data_registro": "31/01/2025",
        "cpf_cnpj_favorecido": "***.999.999-**",
        "favorecido": "FOLHA DE PAGAMENTO DE PESSOAL",

### `GET /consultas/despesas/empenhos/detalhamento.json`
Ficha completa do empenho (classificação, liquidação, pagamento, contrato).

In [ ]:
r = chamar("GET", "/consultas/despesas/empenhos/detalhamento.json", {"id_empenho": ID_EMPENHO, "ano": ANO}, max_linhas=120)

GET /consultas/despesas/empenhos/detalhamento.json -> HTTP 200  (0.2 s, 27,798 bytes, application/json)
params: {'id_empenho': 17475715, 'ano': 2025}

{
  "status": 200,
  "data": {
    "classificacao_orcamentaria": {
      "registros": [
        {
          "id_empenho": 17475715,
          "numero_empenho": 90,
          "ano_exercicio": 2025,
          "data_registro": "31/01/2025",
          "unidade_orcamentaria": "1081 - ADVOCACIA GERAL DO ESTADO",
          "unidade_executora": "1080001 - ADVOCACIA-GERAL DO ESTADO",
          "funcao": "3 - ESSENCIAL A JUSTICA",
          "subfuncao": "122 - ADMINISTRACAO GERAL",
          "programa": "705 - APOIO AS POLITICAS PUBLICAS",
          "acao": "2417 - REMUNERACAO DE PESSOAL ATIVO E ENCARGOS SOCIAIS",
          "categoria_economica": "3 - DESPESAS CORRENTES",
          "grupo_despesa": "1 - PESSOAL E ENCARGOS SOCIAIS",
          "modalidade_aplicacao": "90 - APLICACOES DIRETAS",
          "elemento_despesa": "11 - VENCIMENTOS E VANTAG

### `GET /consultas/despesas/empenhos/detalhamento/download`
O mesmo detalhamento em .xlsx.

In [ ]:
r = chamar("GET", "/consultas/despesas/empenhos/detalhamento/download", {"id_empenho": ID_EMPENHO, "ano": ANO})

GET /consultas/despesas/empenhos/detalhamento/download -> HTTP 200  (0.2 s, 22,354 bytes, application/vnd.openxmlformats-officedocument.spreadsheetml.sheet)
params: {'id_empenho': 17475715, 'ano': 2025}

Arquivo binário (.xlsx/.zip); conteúdo não exibido.


### `GET /consultas/despesas/gerar-csv`
Exportação em CSV por registro (traz função e subfunção). ⚠️ No máximo 31 dias; sem `limit` alto vem truncado.

In [ ]:
r = chamar("GET", "/consultas/despesas/gerar-csv", {"ano": ANO, "periodo_inicio": "0101", "periodo_fim": "3101", "limit": 5})

GET /consultas/despesas/gerar-csv -> HTTP 200  (16.4 s, 6,607 bytes, text/csv; charset=UTF-8)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3101', 'limit': 5}

CSV com 6 linhas. Cabeçalho e primeiras linhas:
"Tipo de Registro";Ano;"Número do Empenho";"Data Registro do Empenho";"Tipo Empenho";"CNPJ/CPF Favorecido";Favorecido;"Valor Atual do Empenho";"Descrição/Histórico do Empenho";"Código Órgão";"Sigla do Órgão";Órgão;"Código Unidade Executora";"Unidade Executora";"Código UPG";"Unidade de Programação de
"Pagamento em Restos a Pagar";2025;61;18/09/2024;0;***.000.033-**;"BANCO DO BRASIL S.A";0,00;"SEM DESCRICAO";1916;"GDPE - SEF";"GESTAO DA DIVIDA PUBLICA ESTADUAL";1910016;"GESTAO DIV.PUB.ESTADUAL";;;28;"ENCARGOS ESPECIAIS";843;"SERVICO DA DIVIDA INTERNA";705;"APOIO AS POLITICAS PUBLICAS";7886;"GESTA
"Liquidação no Exercício";2025;14;29/01/2025;ESTIMADO;394460000141;"MINISTERIO DA FAZENDA - ORGAO CENTRAL";1119368,59;"APROPRIACAO EMPENHO - JUROS E ENCARGOS DA DIVIDA<br>A

## 3. Despesas: listas auxiliares (POST)

Transformam nome em id. Corpo `{ano, periodo_inicio, periodo_fim, termo, limit, page}`. ⚠️ `termo` com acento recebe 403.

### `POST /consultas/despesas/lista-orgaos.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-orgaos.json", corpo={**P, "termo": "educacao", "limit": 5, "page": 1})

POST /consultas/despesas/lista-orgaos.json -> HTTP 200  (1.1 s, 404 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "educacao", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 2,
    "registros": [
      {
        "id_orgao": 12143,
        "nome": "2281 - FUNDACAO DE EDUCACAO PARA O TRABALHO DE MINAS GERAIS"
      },
      {
        "id_orgao": 12158,
        "nome": "1261 - SECRETARIA DE ESTADO DE EDUCACAO"
      }
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-unidades.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-unidades.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-unidades.json -> HTTP 200  (2.1 s, 707 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 1116,
    "registros": [
      {
        "id_orgao": 38833,
        "nome": "2350033 - ABAETE"
      },
      {
        "id_orgao": 38557,
        "nome": "1400017 - ABM/EXECUTORA"
      },
      {
        "id_orgao": 38415,
        "nome": "1510011 - ACADEMIA DE POLICIA CIVIL"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-programas.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-programas.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-programas.json -> HTTP 200  (0.2 s, 1,392 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 158,
    "registros": [
      {
        "id_programa": 117907,
        "codigo_programa": 726,
        "nome_programa": "726 - ACESSO A JUSTICA",
        "nome_programa_sem_id": "ACESSO A JUSTICA"
      },
      {
        "id_programa": 117825,
        "codigo_programa": 108,
        "nome_programa": "108 - ACESSO A MERCADOS",
        "nome_programa_sem_id": "ACESSO A MERCADOS"
      },
      {
        "id_programa": 117891,
        "codigo_programa": 62,
        "nome_programa": "62 - ACESSO A SERVICOS DE SAUDE",
        "nome_programa_sem_id": "ACESSO A SERVICOS DE SAUDE"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-acoes.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-acoes.json", corpo={**P, "termo": "alimentacao", "limit": 5, "page": 1})

POST /consultas/despesas/lista-acoes.json -> HTTP 200  (0.2 s, 1,467 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "alimentacao", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 5,
    "registros": [
      {
        "id_acao": 76195,
        "codigo_acao": 4364,
        "nome_acao": "4364 - ALIMENTACAO COMPLEMENTAR NA PRIMEIRA INFANCIA",
        "nome_acao_sem_id": "ALIMENTACAO COMPLEMENTAR NA PRIMEIRA INFANCIA"
      },
      {
        "id_acao": 75422,
        "codigo_acao": 2157,
        "nome_acao": "2157 - AUXILIOS ALIMENTACAO, CRECHE, MORADIA, SAUDE E OUTROS AUXILIOS",
        "nome_acao_sem_id": "AUXILIOS ALIMENTACAO, CRECHE, MORADIA, SAUDE E OUTROS AUXILIOS"
      },
      {
        "id_acao": 75685,
        "codigo_acao": 2130,
        "nome_acao": "2130 - MELHORIA NA ALIMENTACAO",
        "nome_acao_sem_id": "MELHORIA NA ALIMENTACAO"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }

### `POST /consultas/despesas/lista-funcoes.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-funcoes.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-funcoes.json -> HTTP 200  (6.8 s, 1,043 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 26,
    "registros": [
      {
        "id": 4709,
        "codigo": 4,
        "descricao": "ADMINISTRACAO",
        "nome": "4 - ADMINISTRACAO"
      },
      {
        "id": 4717,
        "codigo": 20,
        "descricao": "AGRICULTURA",
        "nome": "20 - AGRICULTURA"
      },
      {
        "id": 4699,
        "codigo": 8,
        "descricao": "ASSISTENCIA SOCIAL",
        "nome": "8 - ASSISTENCIA SOCIAL"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-subfuncoes.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-subfuncoes.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-subfuncoes.json -> HTTP 200  (5.4 s, 1,083 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 81,
    "registros": [
      {
        "id": 5095,
        "codigo": 605,
        "descricao": "ABASTECIMENTO",
        "nome": "605 - ABASTECIMENTO"
      },
      {
        "id": 5120,
        "codigo": 61,
        "descricao": "ACAO JUDICIARIA",
        "nome": "61 - ACAO JUDICIARIA"
      },
      {
        "id": 5165,
        "codigo": 31,
        "descricao": "ACAO LEGISLATIVA",
        "nome": "31 - ACAO LEGISLATIVA"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-elementos-despesa.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-elementos-despesa.json", corpo={**P, "termo": "material", "limit": 5, "page": 1})

POST /consultas/despesas/lista-elementos-despesa.json -> HTTP 200  (0.2 s, 540 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "material", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 3,
    "registros": [
      {
        "id_elemento_despesa": 10,
        "nome": "52 - EQUIPAMENTOS E MATERIAL PERMANENTE"
      },
      {
        "id_elemento_despesa": 63,
        "nome": "32 - MATERIAL, BEM OU SERVICO PARA DISTRIBUICAO GRATUITA"
      },
      {
        "id_elemento_despesa": 22,
        "nome": "30 - MATERIAL DE CONSUMO"
      }
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-itens-despesa.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-itens-despesa.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-itens-despesa.json -> HTTP 200  (3.7 s, 733 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 440,
    "registros": [
      {
        "id_item_despesa": 3863,
        "nome": "65 - ABONO - RATEIO DOS RECURSOS DO FUNDEB"
      },
      {
        "id_item_despesa": 3886,
        "nome": "10 - ABONO DE FERIAS"
      },
      {
        "id_item_despesa": 155,
        "nome": "21 - ABONO DE FERIAS"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-categorias-economicas.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-categorias-economicas.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-categorias-economicas.json -> HTTP 200  (9.4 s, 492 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 2,
    "registros": [
      {
        "id": 1,
        "codigo": 3,
        "descricao": "DESPESAS CORRENTES",
        "nome": "3 - DESPESAS CORRENTES"
      },
      {
        "id": 2,
        "codigo": 4,
        "descricao": "DESPESAS DE CAPITAL",
        "nome": "4 - DESPESAS DE CAPITAL"
      }
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-grupos-despesa.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-grupos-despesa.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-grupos-despesa.json -> HTTP 200  (20.9 s, 1,072 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 6,
    "registros": [
      {
        "id": 10,
        "codigo": 6,
        "descricao": "AMORTIZACAO DA DIVIDA",
        "nome": "6 - AMORTIZACAO DA DIVIDA"
      },
      {
        "id": 5,
        "codigo": 5,
        "descricao": "INVERSOES FINANCEIRAS",
        "nome": "5 - INVERSOES FINANCEIRAS"
      },
      {
        "id": 2,
        "codigo": 4,
        "descricao": "INVESTIMENTOS",
        "nome": "4 - INVESTIMENTOS"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-modalidades-aplicacao.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-modalidades-aplicacao.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-modalidades-aplicacao.json -> HTTP 200  (4.9 s, 1,402 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 14,
    "registros": [
      {
        "id": 30,
        "codigo": 95,
        "descricao": "APLICACAO DIRETA A CONTA DE RECURSOS DE QUE TRATAM OS EE 1 E 2 DO ART.",
        "nome": "95 - APLICACAO DIRETA A CONTA DE RECURSOS DE QUE TRATAM OS EE 1 E 2 DO ART."
      },
      {
        "id": 5,
        "codigo": 91,
        "descricao": "APLICACAO DIRETA DECORRENTE DE OPERACOES ENTRE ORGAOS, FUNDOS E ENTIDA",
        "nome": "91 - APLICACAO DIRETA DECORRENTE DE OPERACOES ENTRE ORGAOS, FUNDOS E ENTIDA"
      },
      {
        "id": 12,
        "codigo": 90,
        "descricao": "APLICACOES DIRETAS",
        "nome": "90 - APLICACOES DIRETAS"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-ipu.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-ipu.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-ipu.json -> HTTP 200  (5.4 s, 1,469 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 12,
    "registros": [
      {
        "id": 3,
        "codigo": 1,
        "descricao": "RECURSOS DIRETAMENTE RECEBIDOS",
        "nome": "1 - RECURSOS DIRETAMENTE RECEBIDOS"
      },
      {
        "id": 492,
        "codigo": 0,
        "descricao": "RECURSOS DO ESTADO APLICADOS NOS INVESTIMENTOS RELATIVOS A LC 212/2025   PROPAG",
        "nome": "0 - RECURSOS DO ESTADO APLICADOS NOS INVESTIMENTOS RELATIVOS A LC 212/2025   PROPAG"
      },
      {
        "id": 6,
        "codigo": 2,
        "descricao": "RECURSOS RECEBIDOS DE OUTRA UNIDADE ORCAMENTARIA DO ORCAMENTO FISCAL PARA LIVRE UTILIZACAO",
        "nome": "2 - RECURSOS RECEBIDOS DE OUTRA UNIDADE ORCAMENTARIA DO ORCAMENTO FISCAL PARA LIVRE UTILIZACAO"
      },
      "… (+2 itens)"
   

### `POST /consultas/despesas/lista-upg.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-upg.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-upg.json -> HTTP 200  (5.5 s, 1,433 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 5303,
    "registros": [
      {
        "id": 792870,
        "codigo": 1056,
        "descricao": "AC - GERÊNCIA DE INFRA-ESTRUTURA FÍSICA",
        "nome": "1056 - AC - GERÊNCIA DE INFRA-ESTRUTURA FÍSICA"
      },
      {
        "id": 792886,
        "codigo": 1138,
        "descricao": "AC - NÚCLEO DE ZELADORIA",
        "nome": "1138 - AC - NÚCLEO DE ZELADORIA"
      },
      {
        "id": 792857,
        "codigo": 254,
        "descricao": "AC ADMINISTRAÇÃO DE MATERIAIS, ADMINISTRAÇÃO DE FORNECEDORES",
        "nome": "254 - AC ADMINISTRAÇÃO DE MATERIAIS, ADMINISTRAÇÃO DE FORNECEDORES"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-fontes-recurso.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-fontes-recurso.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-fontes-recurso.json -> HTTP 200  (5.6 s, 1,697 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 92,
    "registros": [
      {
        "id": 781,
        "codigo": 73,
        "descricao": "ACORDOS, AJUSTES E DEMAIS INSTRUMENTOS DE COOPERACAO MUTUA COM A UNIAO E SUAS ENTIDADES",
        "nome": "73 - ACORDOS, AJUSTES E DEMAIS INSTRUMENTOS DE COOPERACAO MUTUA COM A UNIAO E SUAS ENTIDADES"
      },
      {
        "id": 748,
        "codigo": 74,
        "descricao": "ACORDOS E AJUSTES DE COOPERACAO MUTUA COM AS INSTITUICOES PRIVADAS E OS ORGANISMOS DO EXTERIOR",
        "nome": "74 - ACORDOS E AJUSTES DE COOPERACAO MUTUA COM AS INSTITUICOES PRIVADAS E OS ORGANISMOS DO EXTERIOR"
      },
      {
        "id": 742,
        "codigo": 66,
        "descricao": "ACORDOS E AJUSTES DE COOPERACAO MUTUA COM OS ESTADOS E O DISTRITO FEDERAL",

### `POST /consultas/despesas/lista-fontes-recurso-stn.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-fontes-recurso-stn.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-fontes-recurso-stn.json -> HTTP 200  (4.8 s, 1,851 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 49,
    "registros": [
      {
        "id": 86,
        "codigo": 605,
        "descricao": "ASSISTÊNCIA FINANCEIRA DA UNIÃO DESTINADA À COMPLEMENTAÇÃO AO PAGAMENTO DOS PISOS SALARIAIS PARA PROFISSIONAIS DA ENFERMAGEM.",
        "nome": "605 - ASSISTÊNCIA FINANCEIRA DA UNIÃO DESTINADA À COMPLEMENTAÇÃO AO PAGAMENTO DOS PISOS SALARIAIS PARA PROFISSIONAIS DA ENFERMAGEM."
      },
      {
        "id": 95,
        "codigo": 804,
        "descricao": "DEMAIS RECURSOS PREVIDENCIÁRIOS",
        "nome": "804 - DEMAIS RECURSOS PREVIDENCIÁRIOS"
      },
      {
        "id": 52,
        "codigo": 711,
        "descricao": "DEMAIS TRANSFERÊNCIAS OBRIGATÓRIAS NÃO DECORRENTES DE REPARTIÇÕES DE RECEITAS.",
        "nome": "711 - DEMAIS TRANSFER

### `POST /consultas/despesas/lista-favorecidos.json`
⚠️ lista grande (23 mil). No ano inteiro a busca leva ~30 s e pode estourar o limite do backend (500 'Maximum execution time'); aqui usamos só janeiro

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-favorecidos.json", corpo={**P, "termo": "CEMIG", "periodo_fim": "3101", "limit": 5, "page": 1})

POST /consultas/despesas/lista-favorecidos.json -> HTTP 200  (1.6 s, 964 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3101", "termo": "CEMIG", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 3,
    "registros": [
      {
        "id_favorecido": 953436,
        "nome": "CEMIG DISTRIBUICAO S.A.",
        "cnpj_cpf_favorecido": "06.981.180/0001-16",
        "descricao": "06.981.180/0001-16 - CEMIG DISTRIBUICAO S.A."
      },
      {
        "id_favorecido": 953435,
        "nome": "CEMIG GERACAO E TRANSMISSAO S.A.",
        "cnpj_cpf_favorecido": "06.981.176/0001-58",
        "descricao": "06.981.176/0001-58 - CEMIG GERACAO E TRANSMISSAO S.A."
      },
      {
        "id_favorecido": 930939,
        "nome": "COMPANHIA ENERGETICA DE MINAS GERAIS - CEMIG",
        "cnpj_cpf_favorecido": "17.155.730/0001-64",
        "descricao": "17.155.730/0001-64 - COMPANHIA ENERGETICA DE MINAS GERAIS - CEMIG"
      }
    ],
    "selecao_va

### `POST /consultas/despesas/lista-empenhos.json`
⚠️ exige id_orgao; só números

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-empenhos.json", corpo={**P, "termo": "", "id_orgao": ID_AGE, "limit": 5, "page": 1})

POST /consultas/despesas/lista-empenhos.json -> HTTP 200  (15.6 s, 914 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "id_orgao": 12135, "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 108070,
    "registros": [
      {
        "id": 1,
        "codigo": 1,
        "descricao": 1,
        "nome": "1 - 1"
      },
      {
        "id": 10,
        "codigo": 10,
        "descricao": 10,
        "nome": "10 - 10"
      },
      {
        "id": 100,
        "codigo": 100,
        "descricao": 100,
        "nome": "100 - 100"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-documentos-pagamento.json`
⚠️ exige id_orgao; só números

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-documentos-pagamento.json", corpo={**P, "termo": "", "id_orgao": ID_AGE, "limit": 5, "page": 1})

POST /consultas/despesas/lista-documentos-pagamento.json -> HTTP 200  (0.7 s, 914 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "id_orgao": 12135, "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 115608,
    "registros": [
      {
        "id": 1,
        "codigo": 1,
        "descricao": 1,
        "nome": "1 - 1"
      },
      {
        "id": 10,
        "codigo": 10,
        "descricao": 10,
        "nome": "10 - 10"
      },
      {
        "id": 100,
        "codigo": 100,
        "descricao": 100,
        "nome": "100 - 100"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-processos-compra.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-processos-compra.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-processos-compra.json -> HTTP 200  (6.5 s, 1,849 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 33649,
    "registros": [
      {
        "id": 1467280,
        "codigo": "0511023 000044/2025",
        "descricao": "\t\t\t\t\t\t\t\r\nCOMPRA DE MEDICAMENTOS PELA A UNIDADE\t\t\t\t\t\t\r\n\t\t\t\t\t\t\t\r\n",
        "nome": "0511023 000044/2025 - \t\t\t\t\t\t\t\r\nCOMPRA DE MEDICAMENTOS PELA A UNIDADE\t\t\t\t\t\t\r\n\t\t\t\t\t\t\t\r\n"
      },
      {
        "id": 1475586,
        "codigo": "2301520 000007/2025",
        "descricao": "- Alargamento da Ponte sobre o Rio Paraibuna Rodovia LMG-874 Dimensão(35,2 x 10,30)m",
        "nome": "2301520 000007/2025 - - Alargamento da Ponte sobre o Rio Paraibuna Rodovia LMG-874 Dimensão(35,2 x 10,30)m"
      },
      {
        "id": 1475100,
        "codigo": "0510037 000309/2025",
     

### `POST /consultas/despesas/lista-procedimentos-contratacao.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-procedimentos-contratacao.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-procedimentos-contratacao.json -> HTTP 200  (4.1 s, 1,210 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 12,
    "registros": [
      {
        "id": 423,
        "codigo": "ADESAO A CONTRATOS CORPORATIVOS",
        "descricao": "ADESAO A CONTRATOS CORPORATIVOS",
        "nome": "ADESAO A CONTRATOS CORPORATIVOS - ADESAO A CONTRATOS CORPORATIVOS"
      },
      {
        "id": 291,
        "codigo": "CARTA CONVITE",
        "descricao": "CARTA CONVITE",
        "nome": "CARTA CONVITE - CARTA CONVITE"
      },
      {
        "id": 289,
        "codigo": "CONCORRENCIA",
        "descricao": "CONCORRENCIA",
        "nome": "CONCORRENCIA - CONCORRENCIA"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-objetos-processo.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-objetos-processo.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-objetos-processo.json -> HTTP 200  (7.0 s, 1,849 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 33649,
    "registros": [
      {
        "id": 1467280,
        "codigo": "0511023 000044/2025",
        "descricao": "\t\t\t\t\t\t\t\r\nCOMPRA DE MEDICAMENTOS PELA A UNIDADE\t\t\t\t\t\t\r\n\t\t\t\t\t\t\t\r\n",
        "nome": "0511023 000044/2025 - \t\t\t\t\t\t\t\r\nCOMPRA DE MEDICAMENTOS PELA A UNIDADE\t\t\t\t\t\t\r\n\t\t\t\t\t\t\t\r\n"
      },
      {
        "id": 1475586,
        "codigo": "2301520 000007/2025",
        "descricao": "- Alargamento da Ponte sobre o Rio Paraibuna Rodovia LMG-874 Dimensão(35,2 x 10,30)m",
        "nome": "2301520 000007/2025 - - Alargamento da Ponte sobre o Rio Paraibuna Rodovia LMG-874 Dimensão(35,2 x 10,30)m"
      },
      {
        "id": 1475100,
        "codigo": "0510037 000309/2025",
     

### `POST /consultas/despesas/lista-contratos-convenios.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-contratos-convenios.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-contratos-convenios.json -> HTTP 200  (9.0 s, 923 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 75713,
    "registros": [
      {
        "id": 0,
        "codigo": 0,
        "descricao": 0,
        "nome": "0 - 0"
      },
      {
        "id": 1279,
        "codigo": 1279,
        "descricao": 1279,
        "nome": "1279 - 1279"
      },
      {
        "id": 1333,
        "codigo": 1333,
        "descricao": 1333,
        "nome": "1333 - 1333"
      },
      "… (+2 itens)"
    ],
    "selecao_valida": []
  }
}


### `POST /consultas/despesas/lista-objetos-contrato.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-objetos-contrato.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-objetos-contrato.json -> HTTP 200  (6.4 s, 2,110 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 15048,
    "registros": [
      {
        "id": 619539,
        "codigo": 9389688,
        "descricao": "  Aquisição de ÁGUA MINERAL NATURAL ENGARRAFADA",
        "nome": "9389688 -   Aquisição de ÁGUA MINERAL NATURAL ENGARRAFADA"
      },
      {
        "id": 629007,
        "codigo": 9479090,
        "descricao": "  ARMAZENAMENTO EXTERNO : HD EXTERNO; POLEGADAS: 2,5; CAPACIDADE: 1 TB",
        "nome": "9479090 -   ARMAZENAMENTO EXTERNO : HD EXTERNO; POLEGADAS: 2,5; CAPACIDADE: 1 TB"
      },
      {
        "id": 619021,
        "codigo": 9386330,
        "descricao": "  Celebração de Termo Aditivo de  alteração de titularidade e reajuste do valor do aluguel mensal do Contrato de Locação de Imóvel nº  9.223.367/2019.  Sede da Supe

### `POST /consultas/despesas/lista-titulos-convenio.json`

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-titulos-convenio.json", corpo={**P, "termo": "", "limit": 5, "page": 1})

POST /consultas/despesas/lista-titulos-convenio.json -> HTTP 200  (4.7 s, 2,023 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 7561,
    "registros": [
      {
        "id": 675007,
        "codigo": 9279766,
        "descricao": "       \nMUTUA COOPERACAO PARA VIABILIZAR O COMPUTO DAS MATRICULAS DE ALUNOS NA ESCOLA FAMILIA AGRICOLA PARA FINS DE REPASSE DOS RECURSOS DO\nFUNDEB.",
        "nome": "9279766 -        \nMUTUA COOPERACAO PARA VIABILIZAR O COMPUTO DAS MATRICULAS DE ALUNOS NA ESCOLA FAMILIA AGRICOLA PARA FINS DE REPASSE DOS RECURSOS DO\nFUNDEB."
      },
      {
        "id": 676382,
        "codigo": 9318117,
        "descricao": "  ACOLHER, TRATAR, REINSERIR DEPENDENTES DE ALCOOL, TABACO E OUTRAS DROGAS PARA UMA SOBRIA CONVIVENCIA E FORTALECIMENTO A PREVENCAO DE RECAIDAS.",
        "nome": "9318117 -   ACOLHER, TRATAR, REINSERIR DEPENDENTES DE ALC

## 4. Receitas

Período em **MM** (`01`–`12`), diferente de despesas.

In [ ]:
PR = {"ano": ANO, "periodo_inicio": "01", "periodo_fim": "12"}

### `GET /consultas/receitas/lista-periodo.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-periodo.json")

GET /consultas/receitas/lista-periodo.json -> HTTP 200  (0.2 s, 1,280 bytes, application/json)

{
  "status": 200,
  "anosMeses": {
    "2021": [
      1,
      2,
      3,
      "… (+9 itens)"
    ],
    "2022": [
      1,
      2,
      3,
      "… (+9 itens)"
    ],
    "2023": [
      1,
      2,
      3,
      "… (+9 itens)"
    ],
    "2024": [
      1,
      2,
      3,
      "… (+9 itens)"
    ],
    "2025": [
      1,
      2,
      3,
      "… (+9 itens)"
    ],
    "2026": [
      1,
      2,
      3,
      "… (+7 itens)"
    ]
  }
}


### `GET /consultas/receitas/visao-geral.json`
Receita por órgão.

In [ ]:
r = chamar("GET", "/consultas/receitas/visao-geral.json", {**PR, "limit": 3, "page": 1})

GET /consultas/receitas/visao-geral.json -> HTTP 200  (0.3 s, 3,207 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '01', 'periodo_fim': '12', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "20/10/2026",
    "total_previsto_inicial": 128949778608,
    "total_previsto_atualizado": 129947677602,
    "total_arrecadado": 132672322619.37,
    "total_percentual_arrecadado": 102.1,
    "subtotal_previsto_inicial": 4593022,
    "subtotal_previsto_atualizado": 4593022,
    "subtotal_arrecadado": 2131208.8,
    "subtotal_percentual_arrecadado": 104.43,
    "qt_orgaos": 86,
    "orgaos": [
      {
        "id_orgao": 1081,
        "nome": "ADVOCACIA GERAL DO ESTADO",
        "previsto_inicial": 0,
        "previsto_atualizado": 0,
        "arrecadado": 955487.53,
        "percentual_arrecadado": 0
      },
      {
        "id_orgao": 2431,
        "nome": "AGENCIA DE DESENVOLVIMENTO DA REGIAO METROPOLITANA DE BELO HORIZONTE",
        "previsto_ini

### `GET /consultas/receitas/classificacao-orcamentaria.json`
Receita pela classificação orçamentária.

In [ ]:
r = chamar("GET", "/consultas/receitas/classificacao-orcamentaria.json", {**PR, "limit": 3, "page": 1})

GET /consultas/receitas/classificacao-orcamentaria.json -> HTTP 200  (0.3 s, 8,874 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '01', 'periodo_fim': '12', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "20/10/2026",
    "colunas_disponiveis": {
      "classificacao_receita": {
        "label": "Classificação da Receita",
        "fixa": true,
        "tipo": "classificacao"
      },
      "orgao": {
        "label": "Órgão",
        "tipo": "complementar"
      },
      "desdobramento_3": {
        "label": "Desdobramento 3",
        "tipo": "nivel_classificacao",
        "nivel": 6
      },
      "categoria_economica": {
        "label": "Categoria Econômica",
        "tipo": "nivel_classificacao",
        "nivel": 1
      },
      "tipo_receita": {
        "label": "Tipo de Receita",
        "tipo": "nivel_classificacao",
        "nivel": 7
      },
      "origem": {
        "label": "Origem",
        "tipo": "nivel_classificacao",

### `GET /consultas/receitas/gerar-csv`

In [ ]:
r = chamar("GET", "/consultas/receitas/gerar-csv", {"ano": ANO, "periodo_inicio": "01", "periodo_fim": "01"})

GET /consultas/receitas/gerar-csv -> HTTP 200  (0.4 s, 787,583 bytes, text/csv; charset=UTF-8)
params: {'ano': 2025, 'periodo_inicio': '01', 'periodo_fim': '01'}

CSV com 1248 linhas. Cabeçalho e primeiras linhas:
Ano;Período;Órgão;"Código do Órgão";"Classificação da receita";"Categoria econômica";"Código da Categoria econômica";Origem;"Código da Origem";"Espécie da receita";"Código da Espécie da receita";"Desdobramento 1";"Código do Desdobramento 1";"Desdobramento 2";"Código do Desdobramento 2";"Desdobrament
2025;1;"ASSEMBLEIA LEGISLATIVA DO ESTADO DE MINAS GERAIS";1011;"	1321010101000";"RECEITAS CORRENTES";1;"RECEITA PATRIMONIAL";3;"VALORES MOBILIARIOS";2;"JUROS E CORRECOES MONETARIAS";1;"REMUNERACAO DE DEPOSITOS BANCARIOS";1;"REMUNERACAO DE DEPOSITOS BANCARIOS";0;"REMUNERACAO DEPOSITOS BANCARIOS - PRI
2025;1;"ASSEMBLEIA LEGISLATIVA DO ESTADO DE MINAS GERAIS";1011;"	1361011101000";"RECEITAS CORRENTES";1;"RECEITA PATRIMONIAL";3;"CESSAO DE DIREITOS";6;"CESSAO DE DIREITOS";1;"CESSAO DO 

### `GET /consultas/receitas/lista-orgaos.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-orgaos.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-orgaos.json -> HTTP 200  (0.2 s, 1,213 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id_orgao": 1011,
        "nome": "1011 - ASSEMBLEIA LEGISLATIVA DO ESTADO DE MINAS GERAIS",
        "nome_sem_id": "ASSEMBLEIA LEGISLATIVA DO ESTADO DE MINAS GERAIS"
      },
      {
        "id_orgao": 1021,
        "nome": "1021 - TRIBUNAL DE CONTAS DO ESTADO DE MINAS GERAIS",
        "nome_sem_id": "TRIBUNAL DE CONTAS DO ESTADO DE MINAS GERAIS"
      },
      {
        "id_orgao": 1031,
        "nome": "1031 - TRIBUNAL DE JUSTICA DO ESTADO DE MINAS GERAIS",
        "nome_sem_id": "TRIBUNAL DE JUSTICA DO ESTADO DE MINAS GERAIS"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-classificacao-receita.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-classificacao-receita.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-classificacao-receita.json -> HTTP 200  (0.2 s, 846 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "classificacao_receita_formatada": 1112510101000,
        "classificacao_receita": "1112.51.0.1.01.000"
      },
      {
        "classificacao_receita_formatada": 1112510102000,
        "classificacao_receita": "1112.51.0.1.02.000"
      },
      {
        "classificacao_receita_formatada": 1112510103000,
        "classificacao_receita": "1112.51.0.1.03.000"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-categoria-economica.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-categoria-economica.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-categoria-economica.json -> HTTP 200  (0.2 s, 785 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id_categoria_economica": 1,
        "nome": "1 - RECEITAS CORRENTES",
        "nome_sem_id": "RECEITAS CORRENTES"
      },
      {
        "id_categoria_economica": 2,
        "nome": "2 - RECEITAS DE CAPITAL",
        "nome_sem_id": "RECEITAS DE CAPITAL"
      },
      {
        "id_categoria_economica": 7,
        "nome": "7 - RECEITA INTRAORCAMENTARIA",
        "nome_sem_id": "RECEITA INTRAORCAMENTARIA"
      },
      "… (+1 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-origem.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-origem.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-origem.json -> HTTP 200  (0.2 s, 1,253 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 647,
        "nome": "1 - DEDUCAO REC. - ICMS - PRINC. - COTA PARTE DO ESTADO - CESSAO DE DIREITOS CREDITORIOS - LEI ESTADUAL NO 19.266/10",
        "nome_sem_id": "DEDUCAO REC. - ICMS - PRINC. - COTA PARTE DO ESTADO - CESSAO DE DIREITOS CREDITORIOS - LEI ESTADUAL NO 19.266/10"
      },
      {
        "id": 306,
        "nome": "1 - DEDUCAO REC. - IMPOSTOS, TAXAS E CONTRIBUICOES DE MELHORIA",
        "nome_sem_id": "DEDUCAO REC. - IMPOSTOS, TAXAS E CONTRIBUICOES DE MELHORIA"
      },
      {
        "id": 242,
        "nome": "1 - IMPOSTOS, TAXAS E CONTRIBUICOES DE MELHORIA",
        "nome_sem_id": "IMPOSTOS, TAXAS E CONTRIBUICOES DE MELHORIA"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-especie.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-especie.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-especie.json -> HTTP 200  (0.2 s, 1,085 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 244,
        "nome": "1 - ALIENACAO DE BENS MOVEIS",
        "nome_sem_id": "ALIENACAO DE BENS MOVEIS"
      },
      {
        "id": 481,
        "nome": "1 - AMORTIZACAO DE EMPRESTIMOS",
        "nome_sem_id": "AMORTIZACAO DE EMPRESTIMOS"
      },
      {
        "id": 208,
        "nome": "1 - CONTRIBUICOES SOCIAIS",
        "nome_sem_id": "CONTRIBUICOES SOCIAIS"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-desdobramento-1.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-desdobramento-1.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-desdobramento-1.json -> HTTP 200  (0.2 s, 1,055 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 360,
        "nome": "1 - ALIENACAO DE BENS IMOVEIS",
        "nome_sem_id": "ALIENACAO DE BENS IMOVEIS"
      },
      {
        "id": 98,
        "nome": "1 - ALIENACAO DE TITULOS, VALORES MOBILIARIOS E APLICACOES CONGENERES",
        "nome_sem_id": "ALIENACAO DE TITULOS, VALORES MOBILIARIOS E APLICACOES CONGENERES"
      },
      {
        "id": 129,
        "nome": "1 - AMORTIZACAO DE EMPRESTIMOS",
        "nome_sem_id": "AMORTIZACAO DE EMPRESTIMOS"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-desdobramento-2.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-desdobramento-2.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-desdobramento-2.json -> HTTP 200  (0.2 s, 1,074 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 2823,
        "nome": "1 - ALIENACAO DE BENS IMOVEIS",
        "nome_sem_id": "ALIENACAO DE BENS IMOVEIS"
      },
      {
        "id": 2825,
        "nome": "1 - ALIENACAO DE BENS MOVEIS E SEMOVENTES",
        "nome_sem_id": "ALIENACAO DE BENS MOVEIS E SEMOVENTES"
      },
      {
        "id": 2829,
        "nome": "1 - ALUGUEIS, ARRENDAMENTOS, FOROS, LAUDEMIOS, TARIFAS DE OCUPACAO",
        "nome_sem_id": "ALUGUEIS, ARRENDAMENTOS, FOROS, LAUDEMIOS, TARIFAS DE OCUPACAO"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-desdobramento-3.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-desdobramento-3.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-desdobramento-3.json -> HTTP 200  (0.2 s, 1,206 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 4575,
        "nome": "0 - ALIENACAO DE BENS IMOVEIS",
        "nome_sem_id": "ALIENACAO DE BENS IMOVEIS"
      },
      {
        "id": 4579,
        "nome": "0 - ALIENACAO DE BENS MOVEIS E SEMOVENTES",
        "nome_sem_id": "ALIENACAO DE BENS MOVEIS E SEMOVENTES"
      },
      {
        "id": 4588,
        "nome": "0 - ALIENACAO DE TITULOS, VALORES MOBILIARIOS E APLICACOES CONGENERES PERMANENTES",
        "nome_sem_id": "ALIENACAO DE TITULOS, VALORES MOBILIARIOS E APLICACOES CONGENERES PERMANENTES"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-tipo-receita.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-tipo-receita.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-tipo-receita.json -> HTTP 200  (0.2 s, 1,276 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 2086,
        "nome": "1 - ADICIONAL ICMS - FUNDO ESTADUAL COMBATE POBREZA - FUNDO DE ERRADICACAO DA MISERIA - FEM - PRINCIPAL",
        "nome_sem_id": "ADICIONAL ICMS - FUNDO ESTADUAL COMBATE POBREZA - FUNDO DE ERRADICACAO DA MISERIA - FEM - PRINCIPAL"
      },
      {
        "id": 1951,
        "nome": "1 - ALIEN. TITULOS VALORES MOBIL. APLIC. CONGEN. PERMANENTES - PRINCIPAL",
        "nome_sem_id": "ALIEN. TITULOS VALORES MOBIL. APLIC. CONGEN. PERMANENTES - PRINCIPAL"
      },
      {
        "id": 8300,
        "nome": "1 - ALIENAA A O BENS MA VEIS E SEMOVENTES - PRINCIPAL",
        "nome_sem_id": "ALIENAA A O BENS MA VEIS E SEMOVENTES - PRINCIPAL"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-item-receita.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-item-receita.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-item-receita.json -> HTTP 200  (0.2 s, 1,171 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 9717,
        "nome": "1 - ADICIONAL ICMS - FEM - DA - COTA PARTE DO ESTADO",
        "nome_sem_id": "ADICIONAL ICMS - FEM - DA - COTA PARTE DO ESTADO"
      },
      {
        "id": 9423,
        "nome": "1 - ADICIONAL ICMS - FEM - DA-MJM - COTA PARTE DO ESTADO",
        "nome_sem_id": "ADICIONAL ICMS - FEM - DA-MJM - COTA PARTE DO ESTADO"
      },
      {
        "id": 9715,
        "nome": "1 - ADICIONAL ICMS - FEM - MJM - COTA PARTE DO ESTADO",
        "nome_sem_id": "ADICIONAL ICMS - FEM - MJM - COTA PARTE DO ESTADO"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-subitem-receita.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-subitem-receita.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-subitem-receita.json -> HTTP 200  (0.2 s, 1,174 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 6670,
        "nome": "0 - ADICIONAL ICMS - FEM - DA - COTA PARTE DO ESTADO",
        "nome_sem_id": "ADICIONAL ICMS - FEM - DA - COTA PARTE DO ESTADO"
      },
      {
        "id": 7032,
        "nome": "0 - ADICIONAL ICMS - FEM - DA - COTA PARTE DO FUNDEB",
        "nome_sem_id": "ADICIONAL ICMS - FEM - DA - COTA PARTE DO FUNDEB"
      },
      {
        "id": 5196,
        "nome": "0 - ADICIONAL ICMS - FEM - DA-MJM - COTA PARTE DO ESTADO",
        "nome_sem_id": "ADICIONAL ICMS - FEM - DA-MJM - COTA PARTE DO ESTADO"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-fonte-recurso.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-fonte-recurso.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-fonte-recurso.json -> HTTP 200  (0.2 s, 1,265 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 2013,
        "nome": "1 - CONVENIOS COM A UNIAO E SUAS ENTIDADES - EMENDAS INDIVIDUAIS",
        "nome_sem_id": "CONVENIOS COM A UNIAO E SUAS ENTIDADES - EMENDAS INDIVIDUAIS"
      },
      {
        "id": 2001,
        "nome": "10 - RECURSOS ORDINARIOS - RECURSOS NAO VINCULADOS DE IMPOSTOS",
        "nome_sem_id": "RECURSOS ORDINARIOS - RECURSOS NAO VINCULADOS DE IMPOSTOS"
      },
      {
        "id": 2039,
        "nome": "11 - RECURSOS DE DESVINCULACAO DE RECEITAS",
        "nome_sem_id": "RECURSOS DE DESVINCULACAO DE RECEITAS"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/receitas/lista-fonte-recurso-stn.json`

In [ ]:
r = chamar("GET", "/consultas/receitas/lista-fonte-recurso-stn.json", {"limit": 5, "page": 1})

GET /consultas/receitas/lista-fonte-recurso-stn.json -> HTTP 200  (0.2 s, 1,255 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id": 1,
        "nome": "500 - RECURSOS NÃO VINCULADOS DE IMPOSTOS",
        "nome_sem_id": "RECURSOS NÃO VINCULADOS DE IMPOSTOS"
      },
      {
        "id": 2,
        "nome": "501 - OUTROS RECURSOS NÃO VINCULADOS",
        "nome_sem_id": "OUTROS RECURSOS NÃO VINCULADOS"
      },
      {
        "id": 3,
        "nome": "502 - RECURSOS NÃO VINCULADOS DA COMPENSAÇÃO DE IMPOSTOS.",
        "nome_sem_id": "RECURSOS NÃO VINCULADOS DA COMPENSAÇÃO DE IMPOSTOS."
      },
      "… (+2 itens)"
    ]
  ]
}


## 5. Diárias e viagens

Período em **DDMM**. Os exemplos individuais usam o governador (autoridade pública), para não expor servidores aleatórios.

### `GET /consultas/diarias/lista-periodo.json`

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-periodo.json")

GET /consultas/diarias/lista-periodo.json -> HTTP 200  (0.4 s, 169 bytes, application/json)

{
  "status": 200,
  "data": [
    [
      2021,
      2022,
      2023,
      "… (+3 itens)"
    ]
  ]
}


### `GET /consultas/diarias/lista-tipos.json`

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-tipos.json")

GET /consultas/diarias/lista-tipos.json -> HTTP 200  (0.6 s, 291 bytes, application/json)

{
  "status": 200,
  "data": [
    {
      "tipo_id": null,
      "tipo_nome": "-"
    },
    {
      "tipo_id": 2,
      "tipo_nome": "INTERNACIONAL"
    },
    {
      "tipo_id": 1,
      "tipo_nome": "NACIONAL"
    }
  ]
}


### `GET /consultas/diarias/visao-geral.json`
Diárias por órgão.

In [ ]:
r = chamar("GET", "/consultas/diarias/visao-geral.json", {**P, "limit": 3, "page": 1})

GET /consultas/diarias/visao-geral.json -> HTTP 200  (2.2 s, 1,213 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "20/10/2026",
    "total_valor_empenhado": 148450624,
    "total_valor_liquidado": 148001916,
    "total_valor_pago": 147229555,
    "total_viajantes": 0,
    "qt_orgaos": 65,
    "orgaos": [
      {
        "id_orgao": 1081,
        "nome": "ADVOCACIA GERAL DO ESTADO",
        "viagens": 225,
        "valor_empenhado": 59247.11,
        "valor_liquidado": 59247.11,
        "valor_pago": 59552.61
      },
      {
        "id_orgao": 2431,
        "nome": "AGENCIA DE DESENVOLVIMENTO DA REGIAO METROPOLITANA DE BELO HORIZONTE",
        "viagens": 45,
        "valor_empenhado": 42849.67,
        "valor_liquidado": 42849.67,
        "valor_pago": 44777.42
      },
      {
        "id_orgao": 2461,
        "nome": "AGENCIA DE DESENVOLVIMENTO DA REGIAO M

### `GET /consultas/diarias/destinos.json`
Diárias por destino.

In [ ]:
r = chamar("GET", "/consultas/diarias/destinos.json", {**P, "limit": 3, "page": 1})

GET /consultas/diarias/destinos.json -> HTTP 200  (3.5 s, 1,117 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "20/10/2026",
    "destinos": [
      {
        "id_destino": 49945,
        "nome_destino": "ZELÂNDIA DISTRITO DE SANTA JULIANA",
        "nome_estado": "MINAS GERAIS",
        "nome_pais": "BRASIL",
        "tipo_destino": "NACIONAL",
        "viagens": 2,
        "valor_total": 0
      },
      {
        "id_destino": 42272,
        "nome_destino": "YAMANASHI",
        "nome_estado": "-",
        "nome_pais": "JAPÃO",
        "tipo_destino": "INTERNACIONAL",
        "viagens": 7,
        "valor_total": 0
      },
      {
        "id_destino": 49868,
        "nome_destino": "XONIN DISTRITO DE GOVERNADOR VALADARES",
        "nome_estado": "MINAS GERAIS",
        "nome_pais": "BRASIL",
        "tipo_destino": "NACIONAL",
        "viagens": 19,
      

### `GET /consultas/diarias/individual.json`
Diárias por viajante. `ano`, `periodo_inicio` e `periodo_fim` obrigatórios.

In [ ]:
viajante = chamar("GET", "/consultas/diarias/individual.json", {**P, "viajante": "ROMEU ZEMA NETO", "limit": 3, "page": 1})
HASH_VIAJANTE = next((v["hash"] for v in viajante["data"]["viajantes"] if v["valor_total"]), None)

GET /consultas/diarias/individual.json -> HTTP 200  (0.5 s, 1,226 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'viajante': 'ROMEU ZEMA NETO', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "viajantes": [
      {
        "cpf": "***.061.616-**",
        "nome": "ROMEU ZEMA NETO",
        "cargo_comissao": "GOVERNADOR DO ESTADO",
        "cd_orgao": 1711,
        "orgao": "SECRETARIA DE ESTADO DE COMUNICACAO SOCIAL",
        "valor_total": 0,
        "hash": "1jjq_YTYWQn9RXB7syCYBZizsIO3qPjUPdfLbRChu4WURVoN-Hh3xYuCC6py0yr6R8wFRaAwyyqCTGE4YOEAbADBFYpcU4oe6YZdzpMIkXWSIu-1iWF98KaU6DFjqNO21IsEBw4vGlXWO1IYJT8s5WiUmk9UDCj1FSYudBacbnOFBWKWGMsAGg1bTcl4WZ4g"
      },
      {
        "cpf": "***.061.616-**",
        "nome": "ROMEU ZEMA NETO",
        "cargo_comissao": "GOVERNADOR DO ESTADO",
        "cd_orgao": 1631,
        "orgao": "SECRETARIA GERAL",
        "valor_total": 33035.8,
        "hash": "Qurh5wnvgZ6smm0vjuKvjhhwRFw0RaL5AP

### `GET /consultas/diarias/individual-documentos.json`
Documentos (viagens) de um viajante; usa o `hash` da célula anterior.

In [ ]:
documentos = chamar("GET", "/consultas/diarias/individual-documentos.json", {"hash": HASH_VIAJANTE, "limit": 3, "page": 1})
ID_DOCUMENTO, TIPO_DOCUMENTO = achar(documentos, "id"), achar(documentos, "tipo")
print("\nid/tipo usados na próxima célula:", ID_DOCUMENTO, TIPO_DOCUMENTO)

GET /consultas/diarias/individual-documentos.json -> HTTP 200  (0.2 s, 1,515 bytes, application/json)
params: {'hash': 'Qurh5wnvgZ6smm0vjuKvjhhwRFw0RaL5APOIoVTxjl00Y2ABLjBxXoF6mYl-yRQ09nzc_hZaT4DVTZQNH0Nr0BqAahUdoH1LARJPMcMgeFDxvYW_oCQga-5SHQNbsahHwVPed6Ke3SFHlG93JWWAx3AZc9vUeOMl0s_cWUKEGhTqyfSlmNQhPpmx1E78ds1f', 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "viajante": "ROMEU ZEMA NETO",
    "documentos": [
      {
        "documento": "PCDP-2025-000393/25",
        "data_inicio": "15/12/2025",
        "data_fim": "15/12/2025",
        "cidade_origem": "TEÓFILO OTONI/MINAS GERAIS/BRASIL",
        "cidade_destino": "BELO HORIZONTE/MINAS GERAIS/BRASIL",
        "valor_total": 0,
        "id": 1428755,
        "tipo": "scdp"
      },
      {
        "documento": "PCDP-2025-000393/25",
        "data_inicio": "15/12/2025",
        "data_fim": "15/12/2025",
        "cidade_origem": "BELO HORIZONTE/MINAS GERAIS/BRASIL",
        "cidade_destino": "TEÓFILO OTONI/MINAS GERAIS/BRASI

### `GET /consultas/diarias/individual-detalhamento.json`
Detalhe de um documento; `id` e `tipo` obrigatórios.

In [ ]:
r = chamar("GET", "/consultas/diarias/individual-detalhamento.json", {"id": ID_DOCUMENTO, "tipo": TIPO_DOCUMENTO})

GET /consultas/diarias/individual-detalhamento.json -> HTTP 200  (0.4 s, 3,144 bytes, application/json)
params: {'id': 1428755, 'tipo': 'scdp'}

{
  "status": 200,
  "data": {
    "id": 1428755,
    "tipo": "scdp",
    "detalhamento": {
      "dados_viajante": {
        "viajante": "ROMEU ZEMA NETO",
        "cpf": "***.061.616-**",
        "matricula": 14716310,
        "orgao": "SECRETARIA GERAL",
        "tipo_viajante": "SERVIDOR",
        "cargo_efetivo": "GOVERNADOR DO ESTADO",
        "cargo_comissao": "GOVERNADOR DO ESTADO"
      },
      "dados_viagem": {
        "documento": "000393/25",
        "dt_inicio": "15/12/2025",
        "dt_fim": "15/12/2025",
        "viagem": "NACIONAL",
        "motivo_aberto": "O Sr. Governador viajou para cumprir a seguinte agenda oficial: Participar da Cerimônia de entrega das instalações do Hospital Regional de Teófilo Otoni à administração do Instituto Mário Penna; Vistoriar as instalações do Hospital; Encontro com empresários do Sindicomérc

### `GET /consultas/diarias/gerar-csv`

In [ ]:
r = chamar("GET", "/consultas/diarias/gerar-csv", {"ano": ANO, "periodo_inicio": "0101", "periodo_fim": "0501"})

GET /consultas/diarias/gerar-csv -> HTTP 200  (0.5 s, 625,037 bytes, text/csv; charset=UTF-8)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '0501'}

CSV com 1157 linhas. Cabeçalho e primeiras linhas:
"Ano de Exercício";Mês;"Documento - PCDP";"Data do Empenho";"Documento - Número do Empenho";Viajante;MASP;CPF;"Cargo Efetivo";"Cargo em Comissão";"Órgão de Exercício";"Tipo de Viajante";Órgão;"Código Órgão";"Data de Início";"Data Fim";"Município de Origem";"Estado de Origem";"País de Origem";"Municí
2025;1;001879/25;;;"FABRICIO ALVES DE OLIVEIRA";14697577;***.185.686-**;"INVESTIGADOR DE POLICIA I";"INVESTIGADOR DE POLÍCIA (I/E/0)";"POLICIA CIVIL";SERVIDOR;"POLICIA CIVIL";1511;04/01/25;05/01/25;PATROCÍNIO;"MINAS GERAIS";BRASIL;"MONTE CARMELO/MINAS GERAIS/BRASIL";"MINAS GERAIS";BRASIL;0;"VEICULO 
2025;1;001368/25;;;"DENIO HENRIQUE SANTOS TAVARES";10791820;***.552.786-**;"POLICIAL PENAL";"POLICIAL PENAL";"SECRETARIA DE JUSTICA E SEGURANCA PUBLICA";SERVIDOR;"SECRETARIA DE JUST

### `GET /consultas/diarias/lista-orgaos.json`

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-orgaos.json", {"limit": 5, "page": 1})

GET /consultas/diarias/lista-orgaos.json -> HTTP 200  (0.8 s, 7,884 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "id_orgao": 1081,
        "orgao": "ADVOCACIA GERAL DO ESTADO"
      },
      {
        "id_orgao": 2431,
        "orgao": "AGENCIA RMBH"
      },
      {
        "id_orgao": 2461,
        "orgao": "AGENCIA RMVA"
      },
      "… (+63 itens)"
    ]
  ]
}


### `GET /consultas/diarias/lista-destinos.json`

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-destinos.json", {"termo": "BRASILIA", "limit": 5, "page": 1})

GET /consultas/diarias/lista-destinos.json -> HTTP 200  (0.4 s, 762 bytes, application/json)
params: {'termo': 'BRASILIA', 'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    {
      "destino_id": 50142,
      "destino_nome": "ANGICOS DE MINAS DISTRITO DE BRASILIA DE MINAS/MINAS GERAIS/BRASIL"
    },
    {
      "destino_id": 50371,
      "destino_nome": "RETIRO DE SANTO ANTÔNIO DISTRITO DE BRASILIA DE MINAS/MINAS GERAIS/BRASIL"
    },
    {
      "destino_id": 42282,
      "destino_nome": "BRASÍLIA/DISTRITO FEDERAL/BRASIL"
    },
    "… (+2 itens)"
  ]
}


### `GET /consultas/diarias/lista-viajantes.json`

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-viajantes.json", {"termo": "ZEMA", "limit": 5, "page": 1})

GET /consultas/diarias/lista-viajantes.json -> HTTP 200  (0.4 s, 372 bytes, application/json)
params: {'termo': 'ZEMA', 'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    {
      "viajante_id": 388914,
      "viajante_nome": "ROMEU ZEMA NETO"
    },
    {
      "viajante_id": 370203,
      "viajante_nome": "ROZEMACO RODRIGUES DO NASCIMENTO"
    },
    {
      "viajante_id": 349689,
      "viajante_nome": "ALZEMAR SANTOS COSTA"
    }
  ]
}


### `GET /consultas/diarias/lista-nome-viajante`
Sem `.json` no path, como está no swagger.

In [ ]:
r = chamar("GET", "/consultas/diarias/lista-nome-viajante", {"termo": "ZEMA", "limit": 5, "page": 1})

GET /consultas/diarias/lista-nome-viajante -> HTTP 200  (0.4 s, 392 bytes, application/json)
params: {'termo': 'ZEMA', 'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "nome": "ALZEMAR SANTOS COSTA",
        "hash": "bae180fbac81ec29f96e733a21274e47b43c3c5f950673dec7a69d37167096b8"
      },
      {
        "nome": "ROMEU ZEMA NETO",
        "hash": "66c13c31fabd47a6f8057405592da7ac2387bb69c6b28daa684bc4a0e4628dcd"
      }
    ]
  ]
}


## 6. Remuneração

Por `ano` e `mes`. A primeira célula descobre o último mês carregado. ⚠️ Traz nome e salário de servidores;
os exemplos individuais usam o governador.

### `GET /consultas/remuneracao/lista-anos-meses.json`

In [ ]:
anos_meses = chamar("GET", "/consultas/remuneracao/lista-anos-meses.json")
disponiveis = anos_meses["data"][0]          # {"2025": ["1", "2", ...], "2026": [...]}
ANO_R = int(max(disponiveis, key=int))
MES_R = int(max(disponiveis[str(ANO_R)], key=int))
print("\nano/mês usados nas próximas células:", ANO_R, MES_R)

GET /consultas/remuneracao/lista-anos-meses.json -> HTTP 200  (0.2 s, 919 bytes, application/json)

{
  "status": 200,
  "data": [
    {
      "2021": [
        "1",
        "2",
        "3",
        "… (+8 itens)"
      ],
      "2022": [
        "1"
      ],
      "2023": [
        "9",
        "11"
      ],
      "2025": [
        "1",
        "2",
        "3",
        "… (+9 itens)"
      ],
      "2026": [
        "1",
        "2",
        "3",
        "… (+3 itens)"
      ]
    }
  ]
}

ano/mês usados nas próximas células: 2026 6


### `GET /consultas/remuneracao/orgao.json`
Remuneração por órgão.

In [ ]:
r = chamar("GET", "/consultas/remuneracao/orgao.json", {"ano": ANO_R, "mes": MES_R, "limit": 3, "page": 1})

GET /consultas/remuneracao/orgao.json -> HTTP 200  (0.2 s, 4,546 bytes, application/json)
params: {'ano': 2026, 'mes': 6, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "qt_servidores_totais": 630374,
    "remuneracao_total": 3912123954.76,
    "media_salarial": 6206.04,
    "ativos_qt": 366545,
    "inativos_qt": 263829,
    "data_ultima_carga": "21/08/2026",
    "situacoes": [
      {
        "cd_situacao": 1,
        "situacao": "ATIVO",
        "remuneracao_bruta": 1915088492.12,
        "quantidade": 363690,
        "percentual": 48.95,
        "remuneracao_ano": [
          {
            "mes": 1,
            "valor": 1783730399.8
          },
          {
            "mes": 2,
            "valor": 1727120365.26
          },
          {
            "mes": 3,
            "valor": 1775182654.1
          },
          "… (+3 itens)"
        ]
      },
      {
        "cd_situacao": 2,
        "situacao": "INATIVO",
        "remuneracao_bruta": 1932764022.2,
        "quanti

### `GET /consultas/remuneracao/cargo.json`
Remuneração por cargo.

In [ ]:
r = chamar("GET", "/consultas/remuneracao/cargo.json", {"ano": ANO_R, "mes": MES_R, "limit": 3, "page": 1})

GET /consultas/remuneracao/cargo.json -> HTTP 200  (0.2 s, 1,357 bytes, application/json)
params: {'ano': 2026, 'mes': 6, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "21/08/2026",
    "cargos": [
      {
        "cargo_efetivo_id": 482,
        "cargo_efetivo": "-",
        "cargo_comissao_id": 255,
        "cargo_comissao": "DIRETOR",
        "qt_servidores": 2,
        "remuneracao_bruta": 9995.52,
        "remuneracao_liquida": 12252.27,
        "remuneracao_total": 13507.63,
        "descontos": 0
      },
      {
        "cargo_efetivo_id": 482,
        "cargo_efetivo": "-",
        "cargo_comissao_id": 242,
        "cargo_comissao": "DAI-7",
        "qt_servidores": 57,
        "remuneracao_bruta": 100235.14,
        "remuneracao_liquida": 189936.04,
        "remuneracao_total": 198247.24,
        "descontos": 0
      },
      {
        "cargo_efetivo_id": 482,
        "cargo_efetivo": "-",
        "cargo_comissao_id": 226,
        "cargo_comis

### `GET /consultas/remuneracao/situacao.json`
Servidores por situação.

In [ ]:
r = chamar("GET", "/consultas/remuneracao/situacao.json", {"ano": ANO_R, "mes": MES_R})

GET /consultas/remuneracao/situacao.json -> HTTP 200  (0.2 s, 313 bytes, application/json)
params: {'ano': 2026, 'mes': 6}

{
  "status": 200,
  "data": [
    {
      "cd_situacao": 1,
      "situacao": "ATIVO"
    },
    {
      "cd_situacao": 2,
      "situacao": "INATIVO"
    },
    {
      "cd_situacao": 4,
      "situacao": "DESIGNADO AO SERVICO ATIVO"
    }
  ]
}


### `GET /consultas/remuneracao/individual.json`
Remuneração individual; `ano` e `mes` obrigatórios.

In [ ]:
individual = chamar("GET", "/consultas/remuneracao/individual.json",
                    {"ano": ANO_R, "mes": MES_R, "id_cargo_comissao": 377, "limit": 3, "page": 1})  # 377 = GOVERNADOR DO ESTADO
MASP, HASH_SERVIDOR = achar(individual, "masp"), achar(individual, "hash")

GET /consultas/remuneracao/individual.json -> HTTP 200  (0.2 s, 1,031 bytes, application/json)
params: {'ano': 2026, 'mes': 6, 'id_cargo_comissao': 377, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "21/08/2026",
    "servidores": [
      {
        "servidor_id": 1126175,
        "remuneracao_id": 55894977,
        "nome_servidor": "MATEUS SIMOES DE ALMEIDA",
        "masp": 14888226,
        "admissao": 1,
        "cargo_efetivo_id": 482,
        "cargo_efetivo_nome": "-",
        "cargo_comissao_id": 377,
        "cargo_comissao_nome": "GOVERNADOR DO ESTADO",
        "orgao_id": 251,
        "orgao_nome": "SECRETARIA GERAL DO ESTADO DE MINAS GERAIS",
        "situacao": "ATIVO",
        "remuneracao_bruta": 41845.49,
        "remuneracao_liquida": 28049.94,
        "remuneracao_total": 41845.49,
        "descontos": 0,
        "hash": "aqJcnE_ifWo0KAhqI1n4KR3nCUtNCRgJt8bD_G4zxfeEjspIERIivy3C1iMZi_62Y7mBSImBSild7UB5GvbQ5mzOrzacOGB6aB3m521Ttzk"
      }

### `GET /consultas/remuneracao/detalhamento.json`
Contracheque detalhado; usa `masp` e `hash` da célula anterior.

In [ ]:
r = chamar("GET", "/consultas/remuneracao/detalhamento.json", {"masp": MASP, "hash": HASH_SERVIDOR, "ano": ANO_R, "mes": MES_R})

GET /consultas/remuneracao/detalhamento.json -> HTTP 200  (8.6 s, 1,752 bytes, application/json)
params: {'masp': 14888226, 'hash': 'aqJcnE_ifWo0KAhqI1n4KR3nCUtNCRgJt8bD_G4zxfeEjspIERIivy3C1iMZi_62Y7mBSImBSild7UB5GvbQ5mzOrzacOGB6aB3m521Ttzk', 'ano': 2026, 'mes': 6}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "21/08/2026",
    "detalhamento": [
      {
        "id_remuneracao": 55894977,
        "id_orgao": 251,
        "mes_ano": "06/2026",
        "numero_admissao": 1,
        "masp": 14888226,
        "nome": "MATEUS SIMOES DE ALMEIDA",
        "cpf": "***.709.056-**",
        "situacao_servidor": "ATIVO",
        "comporta_ativo": 1,
        "orgao_origem": "SECRETARIA GERAL DO ESTADO DE MINAS GERAIS",
        "orgao_exercicio": "SECRETARIA GERAL DO ESTADO DE MINAS GERAIS",
        "unidade_exercicio": "GOVERNADORIA",
        "carga_horaria_semanal": 0,
        "cargo_efetivo": "-",
        "simbolo_cargo": 0,
        "nivel_carreira": 0,
        "grau_carreira": 0,
   

### `GET /consultas/remuneracao/jetons.json`
Jetons (participação em conselhos de estatais).

In [ ]:
r = chamar("GET", "/consultas/remuneracao/jetons.json", {"ano": ANO_R, "mes": MES_R, "limit": 3, "page": 1})

GET /consultas/remuneracao/jetons.json -> HTTP 200  (0.2 s, 2,575 bytes, application/json)
params: {'ano': 2026, 'mes': 6, 'limit': 3, 'page': 1}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "21/08/2026",
    "total_pago_jetons": 120319.05,
    "total_empresas": 10,
    "total_servidores": 40,
    "distribuicao_empresas": [
      {
        "mes_ano": "06/2026",
        "nome": "BDMG",
        "valor": 41484.69
      },
      {
        "mes_ano": "06/2026",
        "nome": "CEMIG",
        "valor": 24001.6
      },
      {
        "mes_ano": "06/2026",
        "nome": "CODEMIG",
        "valor": 54832.76
      }
    ],
    "tabela_pagamentos_jetons": [
      {
        "mes_ano": "06/2026",
        "id_servidor": 452728,
        "situacao_servidor": "ATIVO",
        "id_remuneracao": 55465915,
        "recebedor": "ALINE BRANDAO SILVA",
        "masp": 13829593,
        "cargo_efetivo": "-",
        "cargo_em_comissao": "DAD12",
        "empresa_descricao": "Companhia de Dese

### `GET /consultas/remuneracao/listar-arquivos.json`
Arquivos de dados abertos do mês.

In [ ]:
r = chamar("GET", "/consultas/remuneracao/listar-arquivos.json", {"ano": ANO_R, "mes": MES_R})

GET /consultas/remuneracao/listar-arquivos.json -> HTTP 200  (0.2 s, 110 bytes, application/json)
params: {'ano': 2026, 'mes': 6}

{
  "status": 200,
  "data": {
    "data_ultima_carga": "21/08/2026",
    "arquivos": []
  }
}


### `GET /consultas/remuneracao/lista-orgaos.json`

In [ ]:
r = chamar("GET", "/consultas/remuneracao/lista-orgaos.json", {"limit": 5, "page": 1})

GET /consultas/remuneracao/lista-orgaos.json -> HTTP 200  (0.2 s, 760 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "orgao_id": 151,
        "orgao_nome": "ADEMG ADMINISTR ESTADIOS MG"
      },
      {
        "orgao_id": 2,
        "orgao_nome": "ADVOCACIA GERAL DO ESTADO"
      },
      {
        "orgao_id": 209,
        "orgao_nome": "ADVOCACIA GERAL DO ESTADO DE MINAS GERAIS"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/remuneracao/lista-cargos-efetivos.json`

In [ ]:
r = chamar("GET", "/consultas/remuneracao/lista-cargos-efetivos.json", {"limit": 5, "page": 1})

GET /consultas/remuneracao/lista-cargos-efetivos.json -> HTTP 200  (0.2 s, 709 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "cargo_efetivo_id": 482,
        "cargo_efetivo_nome": "-"
      },
      {
        "cargo_efetivo_id": 1,
        "cargo_efetivo_nome": "ADVOGADO AUTARQUICO"
      },
      {
        "cargo_efetivo_id": 389,
        "cargo_efetivo_nome": "AGENTE ADMINISTRATIVO"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/remuneracao/lista-cargos-comissao.json`

In [ ]:
r = chamar("GET", "/consultas/remuneracao/lista-cargos-comissao.json", {"limit": 5, "page": 1})

GET /consultas/remuneracao/lista-cargos-comissao.json -> HTTP 200  (0.2 s, 777 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "cargo_comissao_id": 602,
        "cargo_comissao_nome": "ADMINISTRADOR DE CENTRO SOCIO-EDUCACIONAL"
      },
      {
        "cargo_comissao_id": 1,
        "cargo_comissao_nome": "ADVOGADO GERAL ADJUNTO DO ESTADO"
      },
      {
        "cargo_comissao_id": 2,
        "cargo_comissao_nome": "ADVOGADO GERAL DO ESTADO"
      },
      "… (+2 itens)"
    ]
  ]
}


### `GET /consultas/remuneracao/lista-empresas.json`

In [ ]:
r = chamar("GET", "/consultas/remuneracao/lista-empresas.json", {"limit": 5, "page": 1})

GET /consultas/remuneracao/lista-empresas.json -> HTTP 200  (0.1 s, 2,031 bytes, application/json)
params: {'limit': 5, 'page': 1}

{
  "status": 200,
  "data": [
    [
      {
        "empresa_id": 7,
        "empresa_sigla": "BDMG",
        "empresa_ordem": 1,
        "empresa_nome": "Banco de Desenvolvimento de Minas Gerais",
        "empresa_descricao": "Instituição financeira de desenvolvimento estadual, responsável por fomentar o crescimento econômico e social de Minas Gerais."
      },
      {
        "empresa_id": 8,
        "empresa_sigla": "CEMIG",
        "empresa_ordem": 2,
        "empresa_nome": "Companhia Energética de Minas Gerais",
        "empresa_descricao": "Empresa de energia elétrica, uma das maiores do setor no Brasil, responsável pela geração, transmissão e distribuição de energia no estado."
      },
      {
        "empresa_id": 13,
        "empresa_sigla": "CODEMIG",
        "empresa_ordem": 3,
        "empresa_nome": "Companhia de Desenvolvimento Econômico d

## 7. Armadilhas conhecidas

Chamadas que mostram os comportamentos que o agente precisa evitar.

### Filtro ignorado em silêncio: `visao-geral` com `id_funcao`
Compare os totais com a primeira chamada de `visao-geral`: são iguais, embora o filtro seja de Saúde.

In [ ]:
r = chamar("GET", "/consultas/despesas/visao-geral.json", {**P, "id_funcao": ID_FUNCAO_SAUDE, "limit": 1, "page": 1})

GET /consultas/despesas/visao-geral.json -> HTTP 200  (2.1 s, 2,232 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'id_funcao': 4711, 'limit': 1, 'page': 1}

{
  "status": 200,
  "data": {
    "total_empenhado": 131563975828.04,
    "total_liquidado": 127112592180.39,
    "total_pago": 125459430296.87,
    "total_liquidado_restos_a_pagar": 3350312793.92,
    "total_pago_restos_a_pagar": 4431039657.47,
    "subtotal_empenhado": 1023743951.58,
    "subtotal_liquidado": 1014541849.87,
    "subtotal_pago": 759567117.27,
    "subtotal_liquidado_restos_a_pagar": 2761361.24,
    "subtotal_pago_restos_a_pagar": 14037404.22,
    "qt_orgaos": 87,
    "orgaos": [
      {
        "id_orgao": 12135,
        "cd_orgao": 1081,
        "sigla": "AGE",
        "nome": "ADVOCACIA GERAL DO ESTADO",
        "empenhado": 1023743951.58,
        "liquidado": 1014541849.87,
        "pago": 759567117.27,
        "liquidado_restos_a_pagar": 2761361.24,
        "p

### `pesquisa-avancada` com `ano` em lista → 422

In [ ]:
r = chamar("POST", "/consultas/despesas/pesquisa-avancada.json", corpo={"nivel": 1, "filtros": {**P, "ano": [ANO]}})

POST /consultas/despesas/pesquisa-avancada.json -> HTTP 422  (0.2 s, 206 bytes, application/json)
corpo:  {"nivel": 1, "filtros": {"ano": [2025], "periodo_inicio": "0101", "periodo_fim": "3112"}}

{
  "status": 422,
  "message": "Parâmetros inválidos para consultar as despesas.",
  "errors": {
    "ano": "Informe um ano válido com quatro dígitos."
  },
  "data": []
}


### `pesquisa-avancada` sem `.json` → 404

In [ ]:
r = chamar("POST", "/consultas/despesas/pesquisa-avancada", corpo={"nivel": 1, "filtros": P})

POST /consultas/despesas/pesquisa-avancada -> HTTP 404  (0.0 s, 153 bytes, application/json; charset=UTF-8)
corpo:  {"nivel": 1, "filtros": {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112"}}

{
  "status": "Error.",
  "message": "O Endpoint solicitado não existe para a API informada. Consulte a documentação da API para maiores informações. "
}


### `empenhos` sem órgão → 500

In [ ]:
r = chamar("GET", "/consultas/despesas/empenhos.json", {**P, "limit": 1, "page": 1})

GET /consultas/despesas/empenhos.json -> HTTP 500  (0.2 s, 384 bytes, application/json)
params: {'ano': 2025, 'periodo_inicio': '0101', 'periodo_fim': '3112', 'limit': 1, 'page': 1}

{
  "message": "É obrigatório informar o Órgão ou a Unidade Executora ou 3 caracteres do Favorecido!",
  "url": "/api/consultas/despesas/empenhos.json?ano=2025&amp;periodo_inicio=0101&amp;periodo_fim=3112&amp;limit=1&amp;page=1",
  "code": 500,
  "file": "/web/php/app/src/Model/Table/TbDespesasTable.php",
  "line": 1194
}


### Termo com acento
Em 25/09/2026 o gateway respondia 403 (HTML do firewall). Em 20/10/2026 **não reproduz mais**: responde 200, com o texto em UTF-8 ou escapado. O `client.py` continua removendo acentos por segurança.

In [ ]:
r = chamar("POST", "/consultas/despesas/lista-orgaos.json", corpo={**P, "termo": "educação", "limit": 5, "page": 1})

POST /consultas/despesas/lista-orgaos.json -> HTTP 200  (1.7 s, 404 bytes, application/json)
corpo:  {"ano": 2025, "periodo_inicio": "0101", "periodo_fim": "3112", "termo": "educação", "limit": 5, "page": 1}

{
  "status": 200,
  "data": {
    "total": 2,
    "registros": [
      {
        "id_orgao": 12143,
        "nome": "2281 - FUNDACAO DE EDUCACAO PARA O TRABALHO DE MINAS GERAIS"
      },
      {
        "id_orgao": 12158,
        "nome": "1261 - SECRETARIA DE ESTADO DE EDUCACAO"
      }
    ],
    "selecao_valida": []
  }
}
